# Step 07 — the simpler correction: location and scale only

**Data type: RNA_array** (GSE65391). **Reads:** `step05_site_{A,B,C}.rds`, `step06_site_{A,B,C}.rds`.
**Writes:** `step07_site_{A,B,C}.rds`.

The same idea without empirical Bayes: each site sends a count, mean and variance per gene. The
coordinator pools them. Each site rescales every gene to the pooled mean and pooled within-site
spread. With about 300 samples per site, the shrinkage ComBat adds should matter little. This step
checks that.

In [1]:
source("../src/paths.R")
source("../src/endotypes.R")
source("../src/federation.R")
start_log("07")
own <- lapply(setNames(SITES, SITES), function(s) {
  d <- readRDS(site_file("05", s))
  send(site_summary(d$E), s, "per-gene n, mean, variance", ncol(d$E))
})
pooled <- pool_summaries(own)
for (s in SITES) {
  d <- readRDS(site_file("05", s))
  d$E <- apply_pooled(d$E, own[[s]], pooled)
  saveRDS(d, site_file("07", s))
}

## How different is it from federated ComBat?

Each site compares its two corrected versions itself and reports one number: the mean absolute
difference on the log2 scale.

In [2]:
diffs <- sapply(SITES, function(s) {
  a <- readRDS(site_file("06", s))$E; b <- readRDS(site_file("07", s))$E
  send(mean(abs(a - b)), s, "mean |ComBat - location/scale|", ncol(a))
})
round(diffs, 4)

A      B      C 
0.0017 0.0020 0.0017

## Findings

The two corrections differ by a few thousandths of a log2 unit per value. With sites this large,
empirical Bayes shrinkage changes almost nothing. Step 08 measures both against the truth.